# Pyro — BiFormer YOLO11-S (PyroNear Solo)

**Goal.** Controlled architectural ablation in the PyroNear-solo chain: replace the
**deepest backbone C3k2 block** of YOLO11-S with a **BiFormer Bi-Level Routing
Attention** block, and measure the effect against the vanilla anchor. The dataset
pipeline and evaluation protocol are **identical** to the Pyro-P1 vanilla reference —
the *only* change is the model architecture.

**Architecture change (the single variable).**
- Backbone **layer 8** — nominal `C3k2, [1024, True]`, which under YOLO11-S width=0.5
  operates at **512 actual channels at P5/32 (20×20)**, immediately before SPPF — is
  replaced by a `BiFormerBlock` (`dim=512, num_heads=8, n_win=4, topk=4, side_dwconv=3`).
- This is a **1-for-1 in-place swap**, so downstream layer indices and `from`-indices are
  unchanged (no graph reference targets layer 8 by absolute index).

**BiFormer source.** `nchwBRA` and `regional_routing_attention_torch` are copied inline
from the BiFormer repo (github.com/rayleizhu/BiFormer); the notebook does **not** import
BiFormer as a package.

**Reference points (printed in Results).**
- Vanilla YOLO11-S unified: recall = 88.52%, FP = 60.74%
- DCT + AG unified (best): recall = 92.05%, FP = 58.49%
- Literature: Saydirasulovich et al. 2023 (BiFormer-YOLOv8): AP = 79.4%, +3.3% over YOLOv8 baseline

**Primary metric.** Image-level binary alert (a frame is flagged positive if *any*
detection exceeds τ). Box-level mAP@0.5 is secondary.

**Hardware.** Kaggle T4 (16 GB).


## 1. Setup

In [1]:
!pip install ultralytics pycocotools matplotlib opencv-python-headless -q

import os, gc, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor, LongTensor

from ultralytics import YOLO

print(f"PyTorch         : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB)       : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.3 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 1.1/1.3 MB 33.9 MB/s eta 0:00:01


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.2 MB ? eta -:--:--


   ━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━ 6.5/12.2 MB 195.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 12.1/12.2 MB 175.8 MB/s eta 0:00:01


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 12.1/12.2 MB 175.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


PyTorch         : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM (GB)       : 15.6


---

## 2. Dataset (pyro-sdis solo) — identical to Pyro-P1 reference

In [2]:
# Dataset target path (same as the vanilla reference pipeline).
PYRO_PATH = "/kaggle/working/pyro_yolo"

print("=" * 64)
print("PyroNear (pyro-sdis) — Pre-computed Dataset Statistics")
print("=" * 64)
print(f"  Path                  : {PYRO_PATH}")
print(f"  Total train boxes     : 28,167")
print(f"  Hard negatives train  : 4,745 images (empty label files)")
print(f"  Mean bbox edge (px)   : 31.8   (at 640x640 input)")
print(f"  Median bbox edge (px) : 24.0")
print(f"  % COCO small (<32px)  : 69.4%")
print(f"  % COCO medium         : 28.5%")
print(f"  % COCO large          : 2.2%")
print()
print("  NOTE: PyroNear has 1 class (smoke). NO fire class. nc=1.")
print("=" * 64)


PyroNear (pyro-sdis) — Pre-computed Dataset Statistics
  Path                  : /kaggle/working/pyro_yolo
  Total train boxes     : 28,167
  Hard negatives train  : 4,745 images (empty label files)
  Mean bbox edge (px)   : 31.8   (at 640x640 input)
  Median bbox edge (px) : 24.0
  % COCO small (<32px)  : 69.4%
  % COCO medium         : 28.5%
  % COCO large          : 2.2%

  NOTE: PyroNear has 1 class (smoke). NO fire class. nc=1.


---

In [3]:
yaml_content = """# pyro-sdis Solo — BiFormer YOLO11-S ablation (data spec identical to Pyro-P1)
path: /kaggle/working/pyro_yolo
train: images/train
val: images/val
nc: 1
names:
  0: smoke
"""

yaml_path = "/kaggle/working/pyro_solo.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"data.yaml written -> {yaml_path}")
print("-" * 64)
print(yaml_content)


data.yaml written -> /kaggle/working/pyro_solo.yaml
----------------------------------------------------------------
# pyro-sdis Solo — BiFormer YOLO11-S ablation (data spec identical to Pyro-P1)
path: /kaggle/working/pyro_yolo
train: images/train
val: images/val
nc: 1
names:
  0: smoke



In [4]:
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

print("Loading PyroNear dataset...")
ds = load_dataset("pyronear/pyro-sdis")
print(f"Train samples : {len(ds['train'])}")
print(f"Val samples   : {len(ds['val'])}")

for split_name, split in [("Train", ds["train"]), ("Val", ds["val"])]:
    has_smoke = sum(1 for s in split if s["annotations"] and s["annotations"].strip())
    no_smoke  = len(split) - has_smoke
    print(f"\n{split_name}:")
    print(f"  Smoke positives  : {has_smoke}")
    print(f"  Hard negatives   : {no_smoke} ({no_smoke/len(split):.1%})")


Loading PyroNear dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/481M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/482M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/29537 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/4099 [00:00<?, ? examples/s]

Train samples : 29537
Val samples   : 4099



Train:
  Smoke positives  : 24792
  Hard negatives   : 4745 (16.1%)



Val:
  Smoke positives  : 3345
  Hard negatives   : 754 (18.4%)


---

In [5]:
import os
from pathlib import Path

PYRO_PATH = "/kaggle/working/pyro_yolo"

PYRO_DIRS = {
    "train_imgs": f"{PYRO_PATH}/images/train",
    "val_imgs":   f"{PYRO_PATH}/images/val",
    "train_lbls": f"{PYRO_PATH}/labels/train",
    "val_lbls":   f"{PYRO_PATH}/labels/val",
}

# 1. Create Directories
for d in PYRO_DIRS.values():
    os.makedirs(d, exist_ok=True)

# 2. Conversion Function
def save_pyro_split(split, img_dir, lbl_dir, split_name):
    for i, sample in enumerate(split):
        img_name = Path(sample["image_name"]).stem
        sample["image"].save(str(Path(img_dir) / f"{img_name}.jpg"))
        ann = sample["annotations"]
        with open(Path(lbl_dir) / f"{img_name}.txt", "w") as f:
            if ann and ann.strip():
                lines = []
                for line in ann.strip().split("\n"):
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, cx, cy, bw, bh = parts
                        lines.append(f"0 {cx} {cy} {bw} {bh}")
                f.write("\n".join(lines))
        if (i + 1) % 5000 == 0:
            print(f"  {split_name}: {i+1} saved...")
    print(f"  {split_name} done: {len(split)} images")

# 3. Execute Conversion
print("Converting PyroNear train...")
save_pyro_split(ds["train"], PYRO_DIRS["train_imgs"], PYRO_DIRS["train_lbls"], "train")
print("Converting PyroNear val...")
save_pyro_split(ds["val"], PYRO_DIRS["val_imgs"], PYRO_DIRS["val_lbls"], "val")


# 4. Integrity Check
print("-- Dataset Integrity Check --------------------------------------------")

img_train = sorted(Path(f"{PYRO_PATH}/images/train").glob("*.*"))
img_val   = sorted(Path(f"{PYRO_PATH}/images/val").glob("*.*"))
lbl_train = sorted(Path(f"{PYRO_PATH}/labels/train").glob("*.txt"))
lbl_val   = sorted(Path(f"{PYRO_PATH}/labels/val").glob("*.txt"))

print(f"  Train images : {len(img_train):>6,}   (expected ~ 24,792)")
print(f"  Train labels : {len(lbl_train):>6,}   (expected ~ 29,537)")
print(f"  Val   images : {len(img_val):>6,}   (expected   4,099)")
print(f"  Val   labels : {len(lbl_val):>6,}   (expected   4,099)")

def is_empty_label(p: Path) -> bool:
    if p.stat().st_size == 0:
        return True
    return p.read_text().strip() == ""

hard_neg_val   = [p for p in lbl_val   if is_empty_label(p)]
hard_neg_train = [p for p in lbl_train if is_empty_label(p)]
n_hard_neg_val = len(hard_neg_val)
n_positive_val = len(img_val) - n_hard_neg_val

print(f"\n  Hard negatives (val)   : {n_hard_neg_val:>6,}   <- FP-rate denominator")
print(f"  Positive images (val)  : {n_positive_val:>6,}   <- Recall denominator")
print(f"  Hard negatives (train) : {len(hard_neg_train):>6,}   (expected ~ 4,745)")

total_train_boxes = 0
for p in lbl_train:
    if p.stat().st_size > 0:
        with open(p) as f:
            total_train_boxes += sum(1 for line in f if line.strip())
print(f"  Total train boxes (counted) : {total_train_boxes:>6,}   (expected ~ 28,167)")

bad_class_files = []
for p in lbl_train[:2000]:
    if p.stat().st_size == 0:
        continue
    for line in p.read_text().splitlines():
        if line.strip() and line.split()[0] != "0":
            bad_class_files.append(p.name)
            break
print(f"  Sample class-id check  : {len(bad_class_files)} files with non-zero class "
      f"(out of 2,000 sampled) -- must be 0")
assert len(bad_class_files) == 0, f"Found non-smoke labels: {bad_class_files[:5]}"

print("\n  Dataset integrity OK")


Converting PyroNear train...


  train: 5000 saved...


  train: 10000 saved...


  train: 15000 saved...


  train: 20000 saved...


...

  train done: 29537 images
Converting PyroNear val...


  val done: 4099 images
-- Dataset Integrity Check --------------------------------------------


  Train images : 29,537   (expected ~ 24,792)
  Train labels : 29,537   (expected ~ 29,537)
  Val   images :  4,099   (expected   4,099)
  Val   labels :  4,099   (expected   4,099)



  Hard negatives (val)   :    754   <- FP-rate denominator
  Positive images (val)  :  3,345   <- Recall denominator
  Hard negatives (train) :  4,745   (expected ~ 4,745)


  Total train boxes (counted) : 28,167   (expected ~ 28,167)
  Sample class-id check  : 0 files with non-zero class (out of 2,000 sampled) -- must be 0

  Dataset integrity OK


---

## 3. BiFormer integration & architecture verification

Five steps, in order:
1. Inline `regional_routing_attention_torch` (from `ops/torch/rrsda.py`).
2. Inline `nchwBRA` (from `ops/bra_nchw.py`), package import removed.
3. `BiFormerBlock` wrapper (LayerNorm in NHWC, attention in NCHW, residual add).
4. Register `BiFormerBlock` via `_CUSTOM` + a `parse_model` patch.
5. Write the modified YOLO11-S YAML (layer-8 swap) and verify the build.


### Step 1 — `regional_routing_attention_torch` (inline from `ops/torch/rrsda.py`)

In [6]:
"""
Regional Routing Scaled Dot-product Attention (based on pytorch API)

author: ZHU Lei
github: https://github.com/rayleizhu
email: ray.leizhu@outlook.com

This source code is licensed under the license found in the
LICENSE file in the root directory of this source tree.
"""
import torch
from torch import Tensor, LongTensor
import torch.nn.functional as F

from typing import Optional, Tuple


def _grid2seq(x:Tensor, region_size:Tuple[int], num_heads:int):
    """
    Args:
        x: BCHW tensor
        region size: int
        num_heads: number of attention heads
    Return:
        out: rearranged x, has a shape of (bs, nhead, nregion, reg_size, head_dim)
        region_h, region_w: number of regions per col/row
    """
    B, C, H, W = x.size()
    region_h, region_w =  H//region_size[0],  W//region_size[1]
    x = x.view(B, num_heads, C//num_heads, region_h, region_size[0], region_w, region_size[1])
    x = torch.einsum('bmdhpwq->bmhwpqd', x).flatten(2, 3).flatten(-3, -2) # (bs, nhead, nregion, reg_size, head_dim)
    return x, region_h, region_w


def _seq2grid(x:Tensor, region_h:int, region_w:int, region_size:Tuple[int]):
    """
    Args: 
        x: (bs, nhead, nregion, reg_size^2, head_dim)
    Return:
        x: (bs, C, H, W)
    """
    bs, nhead, nregion, reg_size_square, head_dim = x.size()
    x = x.view(bs, nhead, region_h, region_w, region_size[0], region_size[1], head_dim)
    x = torch.einsum('bmhwpqd->bmdhpwq', x).reshape(bs, nhead*head_dim,
        region_h*region_size[0], region_w*region_size[1])
    return x


def regional_routing_attention_torch(
    query:Tensor, key:Tensor, value:Tensor, scale:float,
    region_graph:LongTensor, region_size:Tuple[int],
    kv_region_size:Optional[Tuple[int]]=None,
    auto_pad=True)->Tensor:
    """
    Args:
        query, key, value: (B, C, H, W) tensor
        scale: the scale/temperature for dot product attention
        region_graph: (B, nhead, h_q*w_q, topk) tensor, topk <= h_k*w_k
        region_size: region/window size for queries, (rh, rw)
        key_region_size: optional, if None, key_region_size=region_size
        auto_pad: required to be true if the input sizes are not divisible by the region_size
    Return:
        output: (B, C, H, W) tensor
        attn: (bs, nhead, q_nregion, reg_size, topk*kv_region_size) attention matrix
    """
    kv_region_size = kv_region_size or region_size
    bs, nhead, q_nregion, topk = region_graph.size()
    
    # Auto pad to deal with any input size 
    q_pad_b, q_pad_r, kv_pad_b, kv_pad_r = 0, 0, 0, 0
    if auto_pad:
        _, _, Hq, Wq = query.size()
        q_pad_b = (region_size[0] - Hq % region_size[0]) % region_size[0]
        q_pad_r = (region_size[1] - Wq % region_size[1]) % region_size[1]
        if (q_pad_b > 0 or q_pad_r > 0):
            query = F.pad(query, (0, q_pad_r, 0, q_pad_b)) # zero padding

        _, _, Hk, Wk = key.size()
        kv_pad_b = (kv_region_size[0] - Hk % kv_region_size[0]) % kv_region_size[0]
        kv_pad_r = (kv_region_size[1] - Wk % kv_region_size[1]) % kv_region_size[1]
        if (kv_pad_r > 0 or kv_pad_b > 0):
            key = F.pad(key, (0, kv_pad_r, 0, kv_pad_b)) # zero padding
            value = F.pad(value, (0, kv_pad_r, 0, kv_pad_b)) # zero padding
    
    # to sequence format, i.e. (bs, nhead, nregion, reg_size, head_dim)
    query, q_region_h, q_region_w = _grid2seq(query, region_size=region_size, num_heads=nhead)
    key, _, _ = _grid2seq(key, region_size=kv_region_size, num_heads=nhead)
    value, _, _ = _grid2seq(value, region_size=kv_region_size, num_heads=nhead)

    # gather key and values.
    # TODO: is seperate gathering slower than fused one (our old version) ?
    # torch.gather does not support broadcasting, hence we do it manually
    bs, nhead, kv_nregion, kv_region_size, head_dim = key.size()
    broadcasted_region_graph = region_graph.view(bs, nhead, q_nregion, topk, 1, 1).\
        expand(-1, -1, -1, -1, kv_region_size, head_dim)
    key_g = torch.gather(key.view(bs, nhead, 1, kv_nregion, kv_region_size, head_dim).\
        expand(-1, -1, query.size(2), -1, -1, -1), dim=3,
        index=broadcasted_region_graph) # (bs, nhead, q_nregion, topk, kv_region_size, head_dim)
    value_g = torch.gather(value.view(bs, nhead, 1, kv_nregion, kv_region_size, head_dim).\
        expand(-1, -1, query.size(2), -1, -1, -1), dim=3,
        index=broadcasted_region_graph) # (bs, nhead, q_nregion, topk, kv_region_size, head_dim)
    
    # token-to-token attention
    # (bs, nhead, q_nregion, reg_size, head_dim) @ (bs, nhead, q_nregion, head_dim, topk*kv_region_size)
    # -> (bs, nhead, q_nregion, reg_size, topk*kv_region_size)
    # TODO: mask padding region
    attn = (query * scale) @ key_g.flatten(-3, -2).transpose(-1, -2)
    attn = torch.softmax(attn, dim=-1)
    # (bs, nhead, q_nregion, reg_size, topk*kv_region_size) @ (bs, nhead, q_nregion, topk*kv_region_size, head_dim)
    # -> (bs, nhead, q_nregion, reg_size, head_dim)
    output = attn @ value_g.flatten(-3, -2)

    # to BCHW format
    output = _seq2grid(output, region_h=q_region_h, region_w=q_region_w, region_size=region_size)

    # remove paddings if needed
    if auto_pad and (q_pad_b > 0 or q_pad_r > 0):
        output = output[:, :, :Hq, :Wq]

    return output, attn


### Step 2 — `nchwBRA` (inline from `ops/bra_nchw.py`, package import removed)

In [7]:
"""
Refactored Bi-level Routing Attention that takes NCHW input.

author: ZHU Lei
github: https://github.com/rayleizhu
email: ray.leizhu@outlook.com

This source code is licensed under the license found in the
LICENSE file in the root directory of this source tree.
"""
from typing import List, Optional

import torch.nn as nn
import torch
import torch.nn.functional as F
from torch import LongTensor, Tensor

# NOTE: `regional_routing_attention_torch` is defined inline in the previous cell
# (Step 1). The original package import `from ops.torch.rrsda import ...` has been
# removed so this notebook is fully self-contained — no BiFormer package install.

class nchwBRA(nn.Module):
    """Bi-Level Routing Attention that takes nchw input

    Compared to legacy version, this implementation:
    * removes unused args and components
    * uses nchw input format to avoid frequent permutation

    When the size of inputs is not divisible by the region size, there is also a numerical difference
    than legacy implementation, due to:
    * different way to pad the input feature map (padding after linear projection)
    * different pooling behavior (count_include_pad=False)

    Current implementation is more reasonable, hence we do not keep backward numerical compatiability
    """
    def __init__(self, dim, num_heads=8, n_win=7, qk_scale=None, topk=4,  side_dwconv=3, auto_pad=False, attn_backend='torch'):
        super().__init__()
        # local attention setting
        self.dim = dim
        self.num_heads = num_heads
        assert self.dim % num_heads == 0, 'dim must be divisible by num_heads!'
        self.head_dim = self.dim // self.num_heads
        self.scale = qk_scale or self.dim ** -0.5 # NOTE: to be consistent with old models.

        ################side_dwconv (i.e. LCE in Shunted Transformer)###########
        self.lepe = nn.Conv2d(dim, dim, kernel_size=side_dwconv, stride=1, padding=side_dwconv//2, groups=dim) if side_dwconv > 0 else \
                    lambda x: torch.zeros_like(x)
        
        ################ regional routing setting #################
        self.topk = topk
        self.n_win = n_win  # number of windows per row/col

        ##########################################

        self.qkv_linear = nn.Conv2d(self.dim, 3*self.dim, kernel_size=1)
        self.output_linear = nn.Conv2d(self.dim, self.dim, kernel_size=1)

        if attn_backend == 'torch':
            self.attn_fn = regional_routing_attention_torch
        else:
            raise ValueError('CUDA implementation is not available yet. Please stay tuned.')

    def forward(self, x:Tensor, ret_attn_mask=False):
        """
        Args:
            x: NCHW tensor, better to be channel_last (https://pytorch.org/tutorials/intermediate/memory_format_tutorial.html)
        Return:
            NCHW tensor
        """
        N, C, H, W = x.size()
        region_size = (H//self.n_win, W//self.n_win)

        # STEP 1: linear projection
        qkv = self.qkv_linear.forward(x) # ncHW
        q, k, v = qkv.chunk(3, dim=1) # ncHW
       
        # STEP 2: region-to-region routing
        # NOTE: ceil_mode=True, count_include_pad=False = auto padding
        # NOTE: gradients backward through token-to-token attention. See Appendix A for the intuition.
        q_r = F.avg_pool2d(q.detach(), kernel_size=region_size, ceil_mode=True, count_include_pad=False)
        k_r = F.avg_pool2d(k.detach(), kernel_size=region_size, ceil_mode=True, count_include_pad=False) # nchw
        q_r:Tensor = q_r.permute(0, 2, 3, 1).flatten(1, 2) # n(hw)c
        k_r:Tensor = k_r.flatten(2, 3) # nc(hw)
        a_r = q_r @ k_r # n(hw)(hw), adj matrix of regional graph
        _, idx_r = torch.topk(a_r, k=self.topk, dim=-1) # n(hw)k long tensor
        idx_r:LongTensor = idx_r.unsqueeze_(1).expand(-1, self.num_heads, -1, -1) 

        # STEP 3: token to token attention (non-parametric function)
        output, attn_mat = self.attn_fn(query=q, key=k, value=v, scale=self.scale,
                                        region_graph=idx_r, region_size=region_size
                                       )
        
        output = output + self.lepe(v) # ncHW
        output = self.output_linear(output) # ncHW

        if ret_attn_mask:
            return output, attn_mat

        return output


### Step 3 — `BiFormerBlock` wrapper (drop-in YOLO module)

In [8]:
class BiFormerBlock(nn.Module):
    def __init__(self, dim, num_heads=8, n_win=4, topk=4,
                 side_dwconv=3, auto_pad=True):
        super().__init__()
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.attn = nchwBRA(dim=dim, num_heads=num_heads, n_win=n_win,
                            topk=topk, side_dwconv=side_dwconv,
                            auto_pad=auto_pad, attn_backend='torch')

    def forward(self, x):
        # x: NCHW
        # Apply norm in NHWC space, attention in NCHW space
        shortcut = x
        x_nhwc = x.permute(0, 2, 3, 1)
        x_nhwc = self.norm(x_nhwc)
        x = x_nhwc.permute(0, 3, 1, 2)
        x = self.attn(x)
        return x + shortcut


### Step 4 — Register `BiFormerBlock` in Ultralytics (`_CUSTOM` + `parse_model` patch)

We inject `BiFormerBlock` into the `ultralytics.nn.tasks` module namespace so that
`parse_model`'s `globals()[module_name]` lookup resolves the YAML string, and we wrap
`parse_model` to re-inject on every call (robust across Ultralytics versions).

`BiFormerBlock` is **not** added to the channel-handling module sets, so it falls through
`parse_model`'s default branch: its YAML args `[512, 8, 4, 4, 3]` are passed **literally**
(so `dim=512` is *not* width-scaled) and its output channel count is taken as
`c2 = ch[f] = 512` (the block preserves channels). This is exactly what the layer-8 swap
needs at P5/32 under YOLO11-S width=0.5.

In [9]:
import ultralytics.nn.tasks as _tasks

_CUSTOM = {'BiFormerBlock': BiFormerBlock}

# (a) Make the custom module visible in the tasks module namespace.
for _name, _cls in _CUSTOM.items():
    setattr(_tasks, _name, _cls)

# (b) Patch parse_model so custom modules are guaranteed present at parse time.
if not getattr(_tasks.parse_model, "_biformer_patched", False):
    _orig_parse_model = _tasks.parse_model

    def _patched_parse_model(*args, **kwargs):
        g = _tasks.__dict__
        for _name, _cls in _CUSTOM.items():
            g[_name] = _cls
        return _orig_parse_model(*args, **kwargs)

    _patched_parse_model._biformer_patched = True
    _tasks.parse_model = _patched_parse_model

print("Custom modules registered:", list(_CUSTOM.keys()))
print("parse_model patched:", getattr(_tasks.parse_model, "_biformer_patched", False))


Custom modules registered: ['BiFormerBlock']
parse_model patched: True


### Step 5 — Modified YOLO11-S YAML (layer-8 C3k2 → BiFormerBlock)

Backbone **layer 8** `[-1, 2, C3k2, [1024, True]]` is replaced by
`[-1, 1, BiFormerBlock, [512, 8, 4, 4, 3]]` (`[dim, num_heads, n_win, topk, side_dwconv]`).
Because this is a 1-for-1 in-place swap and no node references layer 8 by absolute index,
**every other layer index and `from`-index is unchanged**. `nc: 1` (smoke only).

`n_win=4` (not the default 7): at 640×640 the P5/32 features are 20×20; 7 does not divide
20, but 4 gives clean 5×5 windows. `auto_pad=True` is set as a safety net.

In [10]:
model_yaml = """# YOLO11-S + BiFormerBlock @ backbone layer 8 (deepest C3k2, 512ch, P5/32)
# Only difference from stock yolo11.yaml: layer 8 C3k2 -> BiFormerBlock. nc=1.
nc: 1
scales:
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]              # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]             # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]      # 2
  - [-1, 1, Conv, [256, 3, 2]]             # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]      # 4
  - [-1, 1, Conv, [512, 3, 2]]             # 5-P4/16
  - [-1, 2, C3k2, [512, True]]             # 6
  - [-1, 1, Conv, [1024, 3, 2]]            # 7-P5/32
  - [-1, 1, BiFormerBlock, [512, 8, 4, 4, 3]]  # 8  <-- was: C3k2, [1024, True]
  - [-1, 1, SPPF, [1024, 5]]               # 9
  - [-1, 2, C2PSA, [1024]]                 # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 11
  - [[-1, 6], 1, Concat, [1]]                   # 12 cat backbone P4
  - [-1, 2, C3k2, [512, False]]                 # 13
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 14
  - [[-1, 4], 1, Concat, [1]]                   # 15 cat backbone P3
  - [-1, 2, C3k2, [256, False]]                 # 16 (P3/8-small)
  - [-1, 1, Conv, [256, 3, 2]]                  # 17
  - [[-1, 13], 1, Concat, [1]]                  # 18 cat head P4
  - [-1, 2, C3k2, [512, False]]                 # 19 (P4/16-medium)
  - [-1, 1, Conv, [512, 3, 2]]                  # 20
  - [[-1, 10], 1, Concat, [1]]                  # 21 cat head P5
  - [-1, 2, C3k2, [1024, True]]                 # 22 (P5/32-large)
  - [[16, 19, 22], 1, Detect, [nc]]             # 23 Detect(P3, P4, P5)
"""

MODEL_YAML = "/kaggle/working/yolo11s_biformer.yaml"
with open(MODEL_YAML, "w") as f:
    f.write(model_yaml)
print(f"Model YAML written -> {MODEL_YAML}")


Model YAML written -> /kaggle/working/yolo11s_biformer.yaml


### Architecture verification — all gates must pass before training

Hard gates (abort on failure): layer 8 is `BiFormerBlock` with `dim=512`; full forward
pass produces shape `(2, 5, 8400)` for `nc=1, imgsz=640, batch=2`; no shape-mismatch
error; no NaN in the first forward pass. The parameter count is reported against the
expected `~9.6-9.8M` range (printed, not a hard gate — a single in-place block swap can
land slightly outside the nominal band depending on the C3k2 it replaced).

In [11]:
# Build the modified architecture (nc=1) and transfer COCO-pretrained weights.
# COLD START FROM COCO: build-from-YAML then .load() transfers all matching layers;
# BiFormerBlock (layer 8) has no counterpart in yolo11s.pt, so it stays randomly init'd.
biformer = YOLO(MODEL_YAML)
biformer.load("yolo11s.pt")

det    = biformer.model           # DetectionModel
layers = list(det.model)

print("=" * 64)
print("BiFormer YOLO11-S ARCHITECTURE CHECK")
print("=" * 64)
print(f"\nTotal layers  : {len(layers)}   (expected 24)")
print(f"Last layer    : {type(layers[-1]).__name__}   (expected Detect)")

print("\nLayer summary:")
print(f"  {'Idx':>4}  {'Type':<28}")
print(f"  {'-'*35}")
for i, layer in enumerate(layers):
    print(f"  [{i:>2d}]  {type(layer).__name__}")

# --- Gate 1: layer 8 is BiFormerBlock with dim=512 ---
blk = det.model[8]
assert type(blk).__name__ == "BiFormerBlock", \
    f"Layer 8 is {type(blk).__name__}, expected BiFormerBlock"
assert blk.attn.dim == 512,   f"BiFormerBlock dim={blk.attn.dim}, expected 512"
assert blk.attn.n_win == 4,   f"BiFormerBlock n_win={blk.attn.n_win}, expected 4"
assert blk.attn.topk == 4,    f"BiFormerBlock topk={blk.attn.topk}, expected 4"

# --- Gate 2/3/4: full forward pass -> shape, no mismatch, no NaN ---
det.eval()
dummy = torch.zeros(2, 3, 640, 640)
with torch.no_grad():
    out = det(dummy)
preds = out[0] if isinstance(out, (list, tuple)) else out
print(f"\nForward pass (batch=2, 640x640, nc=1):")
print(f"  Output shape : {tuple(preds.shape)}")
print(f"  Expected     : (2, 5, 8400)")
assert preds.shape == (2, 5, 8400), f"Unexpected shape: {tuple(preds.shape)}"
assert not torch.isnan(preds).any(), "NaN detected in forward pass!"

# --- Param count (reported against expected band) ---
params_total = sum(p.numel() for p in det.parameters())
params_M     = params_total / 1e6
in_band      = 9.6 <= params_M <= 9.8
print(f"\nTotal parameters : {params_total:>12,}  ({params_M:.2f} M)")
print(f"  Expected band  : ~9.6-9.8 M  ->  {'within' if in_band else 'OUTSIDE (informational)'}")

print("\n" + "=" * 64)
print("ALL HARD GATES PASSED")
print("BiFormerBlock active at backbone stage, dim=512, n_win=4, topk=4")
print("=" * 64)

del biformer, det, dummy, out, preds
gc.collect()
torch.cuda.empty_cache()


Transferred 439/453 items from pretrained weights


BiFormer YOLO11-S ARCHITECTURE CHECK

Total layers  : 24   (expected 24)
Last layer    : Detect   (expected Detect)

Layer summary:
   Idx  Type                        
  -----------------------------------
  [ 0]  Conv
  [ 1]  Conv
  [ 2]  C3k2
  [ 3]  Conv
  [ 4]  C3k2
  [ 5]  Conv
  [ 6]  C3k2
  [ 7]  Conv
  [ 8]  BiFormerBlock
  [ 9]  SPPF
  [10]  C2PSA
  [11]  Upsample
  [12]  Concat
  [13]  C3k2
  [14]  Upsample
  [15]  Concat
  [16]  C3k2
  [17]  Conv
  [18]  Concat
  [19]  C3k2
  [20]  Conv
  [21]  Concat
  [22]  C3k2
  [23]  Detect



Forward pass (batch=2, 640x640, nc=1):
  Output shape : (2, 5, 8400)
  Expected     : (2, 5, 8400)

Total parameters :    9,104,595  (9.10 M)
  Expected band  : ~9.6-9.8 M  ->  OUTSIDE (informational)

ALL HARD GATES PASSED
BiFormerBlock active at backbone stage, dim=512, n_win=4, topk=4


---

## 4. Training — 50 epochs, batch=16, imgsz=640, cls_pw=0.0

In [ ]:
# Rebuild a clean model from the modified YAML and transfer COCO weights, then train.
model = YOLO(MODEL_YAML)
model.load("yolo11s.pt")

train_start = time.time()
results = model.train(
    data      = "/kaggle/working/pyro_solo.yaml",
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,
    device    = 0,
    project   = "/kaggle/working/runs",
    name      = "yolo11s_biformer_pyro",
    patience  = 15,
    optimizer = "auto",
    lr0       = 0.01,
    lrf       = 0.01,
    cls_pw    = 0.0,          # YOLO11 default — DO NOT change
    seed      = 0,
    save      = True,
    plots     = True,
    val       = True,
    workers   = 2,
    exist_ok  = True,
)
train_seconds = time.time() - train_start

RUN_DIR = "/kaggle/working/runs/yolo11s_biformer_pyro"
BEST_PT = f"{RUN_DIR}/weights/best.pt"
print(f"\nTraining complete")
print(f"  Wall time     : {train_seconds/3600:.2f} h")
print(f"  Best weights  -> {BEST_PT}")


---

In [ ]:
RUN_DIR = "/kaggle/working/runs/yolo11s_biformer_pyro"

df = pd.read_csv(f"{RUN_DIR}/results.csv")
df.columns = df.columns.str.strip()

print("-- Per-epoch val metrics ----------------------------------------------")
cols_to_show = [c for c in [
    "epoch",
    "train/box_loss", "val/box_loss",
    "train/cls_loss", "val/cls_loss",
    "metrics/mAP50(B)",
    "metrics/precision(B)",
    "metrics/recall(B)",
] if c in df.columns]
print(df[cols_to_show].to_string(index=False))

best_epoch    = int(df["metrics/mAP50(B)"].idxmax())
epochs_actual = int(df["epoch"].max())
print(f"\nBest val mAP@0.5 : {df.loc[best_epoch, 'metrics/mAP50(B)']:.4f}  "
      f"at epoch {int(df.loc[best_epoch, 'epoch'])}")
print(f"Epochs actually trained : {epochs_actual}   (target 50; early-stop if patience triggered)")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("BiFormer YOLO11-S on pyro-sdis Solo (training curves)", fontsize=13)

axes[0].plot(df["epoch"], df["train/box_loss"], label="train")
axes[0].plot(df["epoch"], df["val/box_loss"],   label="val")
axes[0].set_title("Box Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[0].grid(alpha=0.3)

if "train/cls_loss" in df.columns:
    axes[1].plot(df["epoch"], df["train/cls_loss"], label="train")
    axes[1].plot(df["epoch"], df["val/cls_loss"],   label="val")
axes[1].set_title("Class Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(df["epoch"], df["metrics/mAP50(B)"],     label="mAP@0.5")
axes[2].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
axes[2].plot(df["epoch"], df["metrics/recall(B)"],    label="Recall")
axes[2].axvline(x=int(df.loc[best_epoch, "epoch"]), color="gray",
                linestyle="--", alpha=0.5, label="Best epoch")
axes[2].set_title("Val Metrics"); axes[2].set_xlabel("Epoch"); axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/pyro_biformer_training_curves.png", dpi=120)
plt.show()
print("Training curves saved -> /kaggle/working/pyro_biformer_training_curves.png")


---

## 5. Evaluation

In [ ]:
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

RUN_DIR    = "/kaggle/working/runs/yolo11s_biformer_pyro"
BEST_PT    = f"{RUN_DIR}/weights/best.pt"
eval_model = YOLO(BEST_PT)
print(f"Loaded best checkpoint: {BEST_PT}")


In [ ]:
# ================================================================
# BOX-LEVEL EVAL — pyro-sdis val split (secondary metric)
# ================================================================
print("-- Box-level Evaluation: pyro-sdis VAL split -------------------------")

metrics_val = eval_model.val(
    data     = "/kaggle/working/pyro_solo.yaml",
    split    = "val",
    imgsz    = 640,
    batch    = 16,
    device   = 0,
    plots    = True,
    project  = RUN_DIR,
    name     = "eval_val",
    exist_ok = True,
)

map50_val   = float(metrics_val.box.map50)
map5095_val = float(metrics_val.box.map)
prec_val    = float(metrics_val.box.mp)
rec_val     = float(metrics_val.box.mr)
f1_box      = 2 * prec_val * rec_val / max(prec_val + rec_val, 1e-8)

print(f"\n  mAP@0.5      : {map50_val:.4f}   ({map50_val*100:.2f}%)")
print(f"  mAP@0.5:0.95 : {map5095_val:.4f}   ({map5095_val*100:.2f}%)")
print(f"  Precision    : {prec_val:.4f}")
print(f"  Recall (box) : {rec_val:.4f}")
print(f"  F1 (box)     : {f1_box:.4f}")


In [ ]:
# ================================================================
# IMAGE-LEVEL EVAL — predict once at low conf, derive every tau
# ================================================================
# The max-confidence detection per image is invariant to tau for any
# tau <= predict-time-conf. One pass at conf=0.001 -> evaluate every
# threshold analytically.
print("-- Image-level Evaluation: pyro-sdis VAL split -----------------------")

val_img_dir = Path(f"{PYRO_PATH}/images/val")
val_lbl_dir = Path(f"{PYRO_PATH}/labels/val")
val_imgs    = sorted(val_img_dir.glob("*.*"))

def is_hard_neg(stem: str) -> bool:
    lbl = val_lbl_dir / f"{stem}.txt"
    if not lbl.exists() or lbl.stat().st_size == 0:
        return True
    return lbl.read_text().strip() == ""

img_is_pos = {p.name: (not is_hard_neg(p.stem)) for p in val_imgs}
n_total    = len(val_imgs)
n_positive = sum(img_is_pos.values())
n_hardneg  = n_total - n_positive

print(f"  Total val images   : {n_total:>6,}")
print(f"  Positive images    : {n_positive:>6,}   (recall denominator)")
print(f"  Hard-negative imgs : {n_hardneg:>6,}   (FP-rate denominator)")

print(f"\n  Running predictions at conf=0.001 (single pass)...")
BATCH_SIZE = 200
max_conf_per_img = {}

for i in range(0, len(val_imgs), BATCH_SIZE):
    chunk = [str(p) for p in val_imgs[i:i + BATCH_SIZE]]
    preds = eval_model.predict(
        source  = chunk,
        imgsz   = 640,
        conf    = 0.001,
        device  = 0,
        verbose = False,
    )
    for path_str, r in zip(chunk, preds):
        name = Path(path_str).name
        if r.boxes is not None and len(r.boxes) > 0:
            max_conf_per_img[name] = float(r.boxes.conf.max().item())
        else:
            max_conf_per_img[name] = 0.0
    torch.cuda.empty_cache()
    if (i // BATCH_SIZE + 1) % 5 == 0:
        done = min(i + BATCH_SIZE, len(val_imgs))
        print(f"    Processed {done}/{len(val_imgs)} ...")

assert len(max_conf_per_img) == n_total, \
    f"Prediction count mismatch: {len(max_conf_per_img)} vs {n_total}"
print(f"  Prediction pass complete ({len(max_conf_per_img)} images)")


In [ ]:
# -- Helper: image-level metrics at a given tau --------------------------
def metrics_at_tau(tau: float):
    tp = fp = fn = tn = 0
    for name, conf in max_conf_per_img.items():
        flagged = conf >= tau
        if img_is_pos[name]:
            if flagged: tp += 1
            else:       fn += 1
        else:
            if flagged: fp += 1
            else:       tn += 1
    n_pos = tp + fn
    n_neg = fp + tn
    recall  = tp / max(n_pos, 1)
    fp_rate = fp / max(n_neg, 1)
    prec    = tp / max(tp + fp, 1)
    f1      = 2 * prec * recall / max(prec + recall, 1e-8)
    return {"tau": tau, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "recall": recall, "fp_rate": fp_rate,
            "precision": prec, "f1": f1}

print("-- Confidence Threshold Sweep (image-level) --------------------------")
sweep_taus = [0.10, 0.15, 0.25, 0.35, 0.50]
sweep_rows = [metrics_at_tau(t) for t in sweep_taus]

print(f"\n  {'tau':>6} {'Recall':>10} {'FP rate':>10} {'Precision':>12} {'F1':>8}")
print(f"  {'-'*50}")
for r in sweep_rows:
    print(f"  {r['tau']:>6.2f} {r['recall']*100:>9.2f}% {r['fp_rate']*100:>9.2f}% "
          f"{r['precision']*100:>11.2f}% {r['f1']*100:>7.2f}%")

row_025 = next(r for r in sweep_rows if abs(r["tau"] - 0.25) < 1e-6)
img_recall_025  = row_025["recall"]
img_fp_rate_025 = row_025["fp_rate"]
img_prec_025    = row_025["precision"]
f1_025          = row_025["f1"]

print(f"\n-- Primary Operating Point (tau=0.25) --------------------------------")
print(f"  TP / FP / FN / TN   : {row_025['tp']} / {row_025['fp']} / "
      f"{row_025['fn']} / {row_025['tn']}")
print(f"  Image-level Recall  : {img_recall_025*100:.2f}%")
print(f"  Image-level FP rate : {img_fp_rate_025*100:.2f}%")
print(f"  Image-level Prec    : {img_prec_025*100:.2f}%")
print(f"  Image-level F1      : {f1_025*100:.2f}%")

print("\n-- Optimal F1 Search (tau in [0.01, 0.99], step 0.01) ----------------")
fine_taus = np.arange(0.01, 1.00, 0.01)
fine_rows = [metrics_at_tau(float(t)) for t in fine_taus]
best        = max(fine_rows, key=lambda r: r["f1"])
tau_optimal = float(best["tau"])
f1_optimal  = float(best["f1"])

print(f"  Optimal tau*  : {tau_optimal:.3f}")
print(f"  Recall        : {best['recall']*100:.2f}%")
print(f"  FP rate       : {best['fp_rate']*100:.2f}%")
print(f"  Precision     : {best['precision']*100:.2f}%")
print(f"  F1 (image)    : {f1_optimal*100:.2f}%")


In [ ]:
# -- Threshold-sweep plot ------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle("BiFormer YOLO11-S — Image-level Operating Curves (val split)", fontsize=13)

taus_plot = [r["tau"]     for r in fine_rows]
recs      = [r["recall"]  for r in fine_rows]
fprs      = [r["fp_rate"] for r in fine_rows]
f1s       = [r["f1"]      for r in fine_rows]

axes[0].plot(taus_plot, recs, label="Recall",  color="C0", lw=2)
axes[0].plot(taus_plot, fprs, label="FP rate", color="C3", lw=2)
axes[0].plot(taus_plot, f1s,  label="F1",      color="C2", lw=2)
for t in sweep_taus:
    axes[0].axvline(x=t, color="gray", linestyle=":", alpha=0.4)
axes[0].axvline(x=tau_optimal, color="green", linestyle="--", alpha=0.7,
                label=f"tau* = {tau_optimal:.2f}")
axes[0].set_xlabel("Confidence threshold tau")
axes[0].set_ylabel("Rate")
axes[0].set_title("Recall / FP-rate / F1 vs tau")
axes[0].legend(loc="best")
axes[0].grid(alpha=0.3)

axes[1].plot(fprs, recs, color="C0", lw=2)
axes[1].scatter([row_025["fp_rate"]], [row_025["recall"]], color="C3", s=90,
                zorder=5,
                label=f"tau=0.25 (R={row_025['recall']*100:.1f}%, FP={row_025['fp_rate']*100:.1f}%)")
axes[1].scatter([best["fp_rate"]], [best["recall"]], color="green", s=120,
                marker="*", zorder=5,
                label=f"tau*={tau_optimal:.2f} (F1={f1_optimal*100:.1f}%)")
axes[1].set_xlabel("Image-level FP rate")
axes[1].set_ylabel("Image-level Recall")
axes[1].set_title("ROC-style operating curve")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/pyro_biformer_threshold_sweep.png", dpi=120)
plt.show()
print("Threshold-sweep plot saved -> /kaggle/working/pyro_biformer_threshold_sweep.png")


---

## 6. Results — comparison vs published baselines & literature

In [ ]:
print("\n" + "="*72)
print(" BiFormer YOLO11-S on pyro-sdis SOLO")
print("="*72)

print(f"\n  {'Metric':<34} {'Value':>14}")
print(f"  {'-'*50}")
print(f"  {'Image-level Recall (tau=0.25)':<34} {img_recall_025*100:>13.2f}%")
print(f"  {'Image-level FP rate (tau=0.25)':<34} {img_fp_rate_025*100:>13.2f}%")
print(f"  {'Image-level Precision (tau=0.25)':<34} {img_prec_025*100:>13.2f}%")
print(f"  {'Image-level F1 (tau=0.25)':<34} {f1_025*100:>13.2f}%")
print(f"  {'Image-level F1 (optimal tau*)':<34} {f1_optimal*100:>13.2f}%")
print(f"  {'Optimal tau*':<34} {tau_optimal:>14.3f}")
print(f"  {'-'*50}")
print(f"  {'Box-level mAP@0.5':<34} {map50_val*100:>13.2f}%")
print(f"  {'Box-level mAP@0.5:0.95':<34} {map5095_val*100:>13.2f}%")
print(f"  {'Box-level Precision':<34} {prec_val*100:>13.2f}%")
print(f"  {'Box-level Recall':<34} {rec_val*100:>13.2f}%")
print(f"  {'Box-level F1':<34} {f1_box*100:>13.2f}%")
print("="*72)

print("\n-- Comparison Table --------------------------------------------------")
print("\n  PUBLISHED BASELINES (unified training, different protocol):")
print(f"    {'Vanilla YOLO11-S unified':<34} recall=88.52%, FP=60.74%")
print(f"    {'DCT + AG unified (best)':<34} recall=92.05%, FP=58.49%")

print("\n  LITERATURE (contextual reference):")
print(f"    {'Saydirasulovich et al. 2023':<34} BiFormer-YOLOv8: AP=79.4% (+3.3% over YOLOv8 baseline)")

print("\n  THIS RUN — BiFormer YOLO11-S (solo, pyro-sdis only):")
print(f"    {'BiFormer YOLO11-S pyro-sdis':<34} "
      f"recall={img_recall_025*100:.2f}%, FP={img_fp_rate_025*100:.2f}%, "
      f"F1={f1_025*100:.2f}%")

delta_recall = img_recall_025*100 - 88.52
delta_fp     = img_fp_rate_025*100 - 60.74
mark_r       = "UP" if delta_recall >= 0 else "DOWN"
mark_f       = "DOWN" if delta_fp   <  0 else "UP"   # lower FP is better
print(f"\n  Delta vs Vanilla YOLO11-S unified baseline:")
print(f"    Recall   : {delta_recall:+.2f} pp  {mark_r}  ({'better' if delta_recall>=0 else 'worse'})")
print(f"    FP rate  : {delta_fp:+.2f} pp  {mark_f}  ({'better' if delta_fp<0 else 'worse'} -- lower is better)")
print("="*72)


In [ ]:
# -- Save results JSON (image-level vs box-level explicitly disambiguated) --
biformer_results = {
    # -- Image-level (primary metrics) --
    "img_recall"      : round(img_recall_025,  4),   # tau=0.25
    "img_fp_rate"     : round(img_fp_rate_025, 4),   # tau=0.25
    "f1_025"          : round(f1_025,          4),   # image-level F1 @ tau=0.25
    "f1_optimal"      : round(f1_optimal,      4),   # image-level F1 @ tau*
    "tau_optimal"     : round(tau_optimal,     3),
    "precision_img"   : round(img_prec_025,    4),   # image-level @ tau=0.25
    # -- Box-level (secondary) --
    "precision_box"   : round(prec_val,        4),   # from model.val()
    "recall_box"      : round(rec_val,         4),   # from model.val()
    "map50"           : round(map50_val,       4),
    "map5095"         : round(map5095_val,     4),
    "f1_box"          : round(f1_box,          4),
    # -- Model / run metadata --
    "params_M"        : round(params_M,        2),
    "epochs_actual"   : epochs_actual,
    "n_val_total"     : n_total,
    "n_val_positive"  : n_positive,
    "n_val_hardneg"   : n_hardneg,
    "sweep"           : [
        {"tau": r["tau"],
         "recall":    round(r["recall"],    4),
         "fp_rate":   round(r["fp_rate"],   4),
         "precision": round(r["precision"], 4),
         "f1":        round(r["f1"],        4)}
        for r in sweep_rows
    ],
    "train_seconds"   : round(train_seconds,   1),
    "phase"           : "BiFormer_YOLO11S_pyro_solo",
    "model"           : "YOLO11-S + BiFormerBlock replacing C3k2 at deepest backbone stage "
                        "(512ch, n_win=4, topk=4) -- Saydirasulovich et al. Sensors 2023",
    "dataset"         : "pyro-sdis solo (PyroNear, nc=1 smoke)",
    "seed"            : 0,
    "batch"           : 16,
    "imgsz"           : 640,
    "cls_pw"          : 0.0,
}

json_path = "/kaggle/working/pyro_biformer_results.json"
with open(json_path, "w") as f:
    json.dump(biformer_results, f, indent=2)

print(f"Results JSON saved -> {json_path}\n")
print(json.dumps(biformer_results, indent=2))


---

In [ ]:
# -- Final run summary ---------------------------------------------------
print("\n" + "="*72)
print(" BiFormer YOLO11-S RUN COMPLETE")
print("="*72)
print(f"  Wall time      : {train_seconds/3600:.2f} h")
print(f"  Epochs trained : {epochs_actual}   (target 50)")
print(f"  Results JSON   : /kaggle/working/pyro_biformer_results.json")
print(f"  Best weights   : {BEST_PT}")
print(f"  Plots          : /kaggle/working/pyro_biformer_training_curves.png")
print(f"                 : /kaggle/working/pyro_biformer_threshold_sweep.png")
print("="*72)
